In [10]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
import chromadb
from pathlib import Path
import json
from langchain_community.document_loaders import JSONLoader
from sentence_transformers import SentenceTransformer
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"
#Chunk folders
recursive_1000 = DATA_DIR / "chunks" / "recursive_1000"
rsc = DATA_DIR / "chunks" / "rsc"
semantic_p95 = DATA_DIR / "chunks" / "semantic_p95"

In [12]:
#Database
chroma_client = chromadb.PersistentClient(path=str(DATA_DIR / "chromadb"))
colRecursive = chroma_client.get_or_create_collection(name="recursive_100")
colSemantic = chroma_client.get_or_create_collection(name="semantic_p95")
colRsc = chroma_client.get_or_create_collection(name="rsc")

In [13]:
options = [
    "BAAI/bge-base-en-v1.5",#440MB, 768 dimensions. What Medium tutorial uses. Solid retrieval performance, a step up from MiniLM in quality, still runs fine on CPU.
    "BAAI/bge-m3"# 2.2GB, 1024 dimensions. Top multilingual model with a score of 63.0 on MTEB, supporting 100+ languages.Relevant because corpus has Korean, Latin species names, and English mixed together.
]

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1557.27it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Recursive 1000 Chunking 
all_ids = []
all_texts = []
all_metadatas = []

for json_path in sorted(recursive_1000.glob("*.json")):
    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)
    
    for chunk in doc["chunks"]:
        all_ids.append(f"{doc['filename']}_chunk_{chunk['chunk_id']:03d}")
        all_texts.append(chunk["text"])
        all_metadatas.append({
            "filename": doc["filename"],
            "title": doc.get("title") or "",
            "authors": ", ".join(doc.get("authors") or []),
            "year": doc.get("year") or "",
        })

print(f"Loaded {len(all_ids)} chunks")

In [ ]:
embeddings = model.encode(rec1000_chunks, show_progress_bar=True)